# RQ5 – Distributed federated learning, run D1 (Colab)

Runs the distributed FedAvg experiment with **separate processes**: one Flower SuperLink (server) and one SuperNode per bank (SBI, HDFC, AXIS), each reading only its own data folder. Then runs the simulated FedAvg on the same data and settings and compares the two.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. On your PC, make a zip of the merged `master` **without** the `mydev` folder or old zips. The easiest way, in the repo folder in PowerShell:
   ```
   git checkout master
   git pull
   git archive -o Thesis-Coding-src.zip HEAD
   ```
3. Run the cells in order. Cell 2 asks you to upload `Thesis-Coding-src.zip`.

Total time on a T4: about 45–60 minutes. The last cell downloads `d1_results.zip`; upload that file to Claude.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-f01259e9-c141-61a5-ea2a-87b874a727cf)


## 2. Upload and unzip the repository

In [ ]:
from google.colab import files
import os, pathlib, shutil, zipfile

uploaded = files.upload()                      # choose Thesis-Coding-src.zip
zip_name = next(iter(uploaded))
shutil.rmtree("/content/_unzip", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/_unzip")

# find the repository root (the folder that contains fl/distributed/server_app.py)
hits = [p for p in pathlib.Path("/content/_unzip").rglob("server_app.py")
        if p.parent.name == "distributed" and "mydev" not in p.parts]
assert hits, "fl/distributed/server_app.py not found - zip the merged master (round 8 included)"
repo = hits[0].parents[2]
shutil.rmtree("/content/Thesis-Coding", ignore_errors=True)
shutil.move(str(repo), "/content/Thesis-Coding")
os.chdir("/content/Thesis-Coding")
print("Repository at", os.getcwd())
print(sorted(os.listdir("fl/distributed")))

Saving Thesis-Coding-src.zip to Thesis-Coding-src.zip
Repository at /content/Thesis-Coding
['__init__.py', 'app', 'client_app.py', 'codec.py', 'compare.py', 'prepare.py', 'server_app.py', 'settings.py', 'tiny_model.py']


## 3. Install the requirements (about 3–5 minutes)

In [ ]:
%cd /content/Thesis-Coding
!pip -q install -r requirements.txt "flwr>=1.45,<2"
!flwr --version

/content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
Flower version: 1.45.0


## 4. Split the data: one folder per bank, plus test sets for the server

In [ ]:
%cd /content/Thesis-Coding
!python -m fl.distributed.prepare --out deploy/nodes

/content
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
SBI/train.jsonl                                180 rows  banks=SBI
HDFC/train.jsonl                               180 rows  banks=HDFC
AXIS/train.jsonl                               180 rows  banks=AXIS
server/eval/synthetic_test__SBI.jsonl           12 rows  banks=SBI
server/eval/heldout_style__SBI.jsonl            12 rows  banks=SBI
server/eval/synthetic_test__HDFC.jsonl          12 rows  banks=HDFC
server/eval/heldout_style__HDFC.jsonl           12 rows  banks=HDFC
server/eval/synthetic_test__AXIS.jsonl          12 rows  banks=AXIS
server/eval/heldout_style__AXIS.jsonl           12 rows  banks=AXIS
server/eval/real_thesis.jsonl                    8 rows  banks=HDFC,SBI
server/eval/real_thesis_hard.jsonl               8 rows  banks=HDFC,SBI
server/eval/real_axis.jsonl                      4 rows  banks=AXIS
server/eval/injection_axis.jsonl                 4 row

## 5. Run settings and the run script

D1 = Qwen2.5-0.5B-Instruct, 3 rounds × 8 local steps, batch 1 × gradient accumulation 8 (same effective batch as Run 1), 6 cards per test set.

In [ ]:
%%writefile /content/d1.toml
model-name = "Qwen/Qwen2.5-0.5B-Instruct"
num-server-rounds = 3
local-steps = 8
batch-size = 1
grad-accum = 8
eval-limit = 6
out-dir = "results_fl/distributed_quick"

Writing /content/d1.toml


In [ ]:
%%writefile /content/run_d1.sh
#!/usr/bin/env bash
# usage: bash /content/run_d1.sh <arm> "<banks>" <min-nodes> <run-config>
# Starts a SuperLink (server) and one SuperNode per bank as separate processes,
# submits the run, and waits until the ServerApp has written its results.
set -u
ARM=$1; BANKS=$2; MIN=$3; RC=$4
cd /content/Thesis-Coding
export PYTHONPATH=$PWD PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
L=results_fl/logs_$ARM; mkdir -p $L
export FLWR_HOME=$PWD/$L/flwr_home; mkdir -p $FLWR_HOME
printf '[superlink]\ndefault = "d"\n[superlink.d]\naddress = "127.0.0.1:8000"\ninsecure = true\n' > $FLWR_HOME/config.toml
cp $RC $L/run.toml; printf 'arm = "%s"\nmin-nodes = %s\n' $ARM $MIN >> $L/run.toml
pkill -f flower-super; sleep 3
nohup flower-superlink --insecure --disable-runtime-dependency-installation > $L/superlink.log 2>&1 &
sleep 6
p=9101; for B in $BANKS; do
  nohup flower-supernode --insecure --superlink 127.0.0.1:8000 --port $p \
    --node-config "bank='$B' data-dir='$PWD/deploy/nodes/$B'" > $L/node_$B.log 2>&1 &
  p=$((p+1)); done
sleep 10
flwr run fl/distributed/app d --run-config $L/run.toml 2>&1 | tail -n 2
OUT=results_fl/distributed_quick
echo "waiting for $ARM (checks every 30 s, up to 60 min) ..."
for i in $(seq 1 120); do
  [ -f $OUT/predictions_$ARM.jsonl ] && break
  if grep -qiE "Traceback|OutOfMemory" $L/node_*.log $L/superlink.log; then echo "ERROR - see $L"; break; fi
  sleep 30
done
sleep 30
python - "$OUT/rounds_$ARM.jsonl" <<'PY'
import json, sys
try:
    for line in open(sys.argv[1]):
        r = json.loads(line)
        banks = ", ".join(f"{b} n={v['num_examples']} loss={v['train_loss']:.3f} {v['train_seconds']:.0f}s" for b, v in r["banks"].items())
        print(f"round {r['round']}: {banks} | failures={len(r['failures'])} | {r['bytes_down_per_bank']/1e6:.2f} MB per message | round {r['round_seconds']:.0f}s")
except FileNotFoundError:
    print("no rounds file - the run did not finish")
PY
pkill -f flower-super

Writing /content/run_d1.sh


## 6. Distributed FedAvg – all three banks (about 15–25 minutes)

Working if every round lists AXIS, HDFC and SBI with `failures=0`.

In [ ]:
%cd /content/Thesis-Coding
!rm -rf results_fl/distributed_quick
!bash /content/run_d1.sh fedavg "SBI HDFC AXIS" 3 /content/d1.toml

/content
Using SuperLink: d (127.0.0.1:8000)
Successfully started run 14336244588808906131
waiting for fedavg (checks every 30 s, up to 60 min) ...
round 1: AXIS n=180 loss=0.089 76s, HDFC n=180 loss=0.076 78s, SBI n=180 loss=0.063 79s | failures=0 | 8.65 MB per message | round 143s
round 2: AXIS n=180 loss=0.036 78s, HDFC n=180 loss=0.049 80s, SBI n=180 loss=0.017 78s | failures=0 | 8.65 MB per message | round 147s
round 3: AXIS n=180 loss=0.020 77s, HDFC n=180 loss=0.025 82s, SBI n=180 loss=0.009 80s | failures=0 | 8.65 MB per message | round 146s


If the cell above printed `ERROR` or *no rounds file*, run the next cell and paste its output to Claude before continuing.

In [ ]:
!tail -n 30 /content/Thesis-Coding/results_fl/logs_fedavg/node_*.log /content/Thesis-Coding/results_fl/logs_fedavg/superlink.log

==> /content/Thesis-Coding/results_fl/logs_fedavg/node_AXIS.log <==
INFO :      
INFO :      [RUN 14336244588808906131]
INFO :      Receiving: train message (ID: 886dae0209dd701ed5549c12560d9d32556e317a5c2bd3ad0eca30e78bebb786)
INFO :      Start `flwr-clientapp` process
INFO :      Received successfully
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 344.45it/s]
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
INFO :      
INFO :      [RUN 14336244588808906131]
INFO :      Sending: train message (ID: c5e2e0074f5d826b54fdb7c66dec9840a310252f4b96

## 7. Distributed FedAvg – leave one bank out (Axis node not started; about 10–15 minutes)

In [ ]:
%cd /content/Thesis-Coding
!bash /content/run_d1.sh fedavg_lobo "SBI HDFC" 2 /content/d1.toml

/content
Using SuperLink: d (127.0.0.1:8000)
Successfully started run 4438116738338653576
waiting for fedavg_lobo (checks every 30 s, up to 60 min) ...
round 1: HDFC n=180 loss=0.076 55s, SBI n=180 loss=0.063 54s | failures=0 | 8.65 MB per message | round 99s
round 2: HDFC n=180 loss=0.050 57s, SBI n=180 loss=0.017 53s | failures=0 | 8.65 MB per message | round 102s
round 3: HDFC n=180 loss=0.028 56s, SBI n=180 loss=0.009 54s | failures=0 | 8.65 MB per message | round 105s


## 8. Simulated FedAvg on the same data and settings (about 10–15 minutes)

In [ ]:
%cd /content/Thesis-Coding
!rm -rf results_fl/sim_quick
!PYTHONPATH=$PWD python -m fl.distributed.compare simulate --run-config /content/d1.toml --out results_fl/sim_quick

## 9. Results

In [12]:
%cd /content/Thesis-Coding
!PYTHONPATH=$PWD python -m fl.distributed.compare diff --sim results_fl/sim_quick --dist results_fl/distributed_quick
!echo; cat results_fl/distributed_quick/report.md

/content
# Distributed vs simulated FedAvg (fedavg)

| Test set | simulated | distributed |
|---|---|---|
| synthetic_test/AXIS | 28/30 (0.93) | 28/30 (0.93) |
| synthetic_test/HDFC | 29/30 (0.97) | 29/30 (0.97) |
| synthetic_test/SBI | 29/30 (0.97) | 29/30 (0.97) |
| heldout_style/AXIS | 24/30 (0.80) | 26/30 (0.87) |
| heldout_style/HDFC | 22/30 (0.73) | 22/30 (0.73) |
| heldout_style/SBI | 25/30 (0.83) | 25/30 (0.83) |
| real_thesis | 27/30 (0.90) | 27/30 (0.90) |
| real_thesis_hard | 22/30 (0.73) | 21/30 (0.70) |
| real_axis | 17/20 (0.85) | 17/20 (0.85) |
| injection_axis | 16/20 (0.80) | 16/20 (0.80) |

Cards with the same all-correct outcome: 51/52
Exact McNemar (distributed vs simulated, cards all correct): b=1 c=0 p=1
Largest absolute difference between the two final adapters: 0.00238

Distributed transport: 3 rounds, banks AXIS, HDFC, SBI, 8.65 MB per adapter message, 155.71 MB in total, mean round 145.5 s


# RQ5 – federated rule extractor: field accuracy (correct fields / to

## 10. Download everything and upload `d1_results.zip` to Claude

In [13]:
%cd /content/Thesis-Coding
!zip -qr /content/d1_results.zip results_fl/distributed_quick results_fl/sim_quick results_fl/logs_fedavg results_fl/logs_fedavg_lobo -x "*/flwr_home/*" "*/adapters/*"
from google.colab import files
files.download("/content/d1_results.zip")

/content


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>